# Lab 4A — Recurrent neural networks for forecasting
## STUDENT — guided exercises

**Based on Lecture 4, slides 1–95 and 230–242.** Expected duration: **about 2 hours**.

**Goal:** understand what a SimpleRNN learns and how its input/output shapes, training loss and evaluation change across forecasting tasks.

We will compare a persistence baseline, linear regression, and a SimpleRNN. We will then forecast **10 future samples** in two ways: **direct sequence-to-vector** and **sequence-to-sequence training**. We will also visualize why gradients can vanish or explode in long recurrent chains.

This lab deliberately stops **before LSTM and GRU**. All data are synthetic, reproducible and generated locally; no files, APIs or GPUs are required.

**Programming note:** Almost all code is provided and commented. Follow the numbered questions and modify only the few suggested values. Run cells in order (Kernel → Restart & Run All).

## 0. Setup and experiment rules (5–10 min)

We use **Keras 3**. It can run with a TensorFlow or PyTorch backend; the recurrent layers and `model.fit` API are the same. The cell below selects the PyTorch backend to work even where TensorFlow is unavailable. On Google Colab, `keras` and `torch` are typically preinstalled; if either is missing, run `%pip install keras torch` once.

**Important:** Never choose a model based on test performance. We split **complete independent recordings** into train/validation/test sets *before* creating overlapping windows.

In [ ]:
# Run this cell first. The Keras backend must be selected before importing keras.
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import keras

SEED = 17
rng = np.random.default_rng(SEED)
keras.utils.set_random_seed(SEED)

print("Keras version:", keras.__version__)
print("Backend:", keras.backend.backend())
print("NumPy version:", np.__version__)

## 1. A temporal dataset: one sample is a sequence (10–15 min)

Imagine many independent runs of a process sensor. Each run has oscillatory components, a slow change, and measurement noise. Parameters change between runs to mimic different operating conditions. **These signals are teaching examples, not measured industrial data.**

Let the input contain **L = 50** past samples and let the forecast contain **H = 10** future samples:

$$\mathbf X^{(i)}\in\mathbb R^{L\times d},\qquad d=1.$$ 

In a batch of N windows, the input tensor has shape **[N, L, d]**. For direct forecasting, the target has shape **[N, H]**.

In [ ]:
# Each recording is generated independently; we never split one recording
# between training and test, even though its windows will overlap.
RECORD_LENGTH = 180
N_TRAIN, N_VAL, N_TEST = 16, 4, 4
L = 50                  # input history length (slides 230–242)
H = 10                  # how many future values to forecast
STRIDE = 3              # step between consecutive windows within each recording

def generate_recording(generator, length=RECORD_LENGTH):
    """Make one slightly different, noisy process-sensor recording."""
    t = np.arange(length, dtype=np.float32)
    phase = generator.uniform(0, 2 * np.pi)
    amplitude = generator.uniform(0.75, 1.25)
    speed = generator.uniform(0.105, 0.145)    # slightly different cycle rates
    slow = amplitude * np.sin(speed * t + phase)
    faster = 0.30 * np.sin(2.2 * speed * t + 0.45 * phase)
    drift = 0.20 * np.cos(0.035 * t + generator.uniform(-1, 1))
    noise = generator.normal(0, 0.035, size=length)
    return (slow + faster + drift + noise).astype("float32")

recordings = [generate_recording(rng) for _ in range(N_TRAIN + N_VAL + N_TEST)]
train_records = recordings[:N_TRAIN]
val_records = recordings[N_TRAIN:N_TRAIN+N_VAL]
test_records = recordings[N_TRAIN+N_VAL:]

plt.figure(figsize=(10, 3))
for k in range(3):
    plt.plot(recordings[k][:110], label=f"recording {k+1}")
plt.xlabel("Sample index")
plt.ylabel("Sensor value")
plt.title("Independent synthetic process recordings")
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

We construct three aligned targets for every window:

- **One-step target:** the very next value, shape `[N, 1]`.
- **Direct 10-step target:** the next 10 values after the window, shape `[N, 10]`.
- **Sequence-to-sequence 10-step target:** at **each of the L input positions**, the next 10 values relative to that position, shape `[N, L, 10]`.

The last position of the sequence-to-sequence target is **exactly equal** to the direct 10-step target. This is essential for fair evaluation.

In [ ]:
def make_windows(series_list, history=L, horizon=H, stride=STRIDE):
    """Convert full recordings into supervised examples without mixing splits.

    X      : [N, history, 1], the observed history.
    Y10    : [N, horizon], the future after the FINAL observed step.
    Y_each : [N, history, horizon], future from EVERY observed step.
    """
    X_list, Y10_list, Yeach_list = [], [], []
    for series in series_list:
        last_start = len(series) - history - horizon
        for start in range(0, last_start + 1, stride):
            # The input ends at index start + history - 1.
            observed = series[start:start + history]
            future_final = series[start + history:start + history + horizon]

            # Target at input position j forecasts positions j+1, ..., j+horizon.
            future_at_each_step = np.stack([
                series[start + j + 1:start + j + 1 + horizon]
                for j in range(history)
            ])
            X_list.append(observed[:, None])  # add the channel dimension d=1
            Y10_list.append(future_final)
            Yeach_list.append(future_at_each_step)

    return (np.asarray(X_list, dtype="float32"),
            np.asarray(Y10_list, dtype="float32"),
            np.asarray(Yeach_list, dtype="float32"))

X_train, Y10_train, Yseq_train = make_windows(train_records)
X_val, Y10_val, Yseq_val = make_windows(val_records)
X_test, Y10_test, Yseq_test = make_windows(test_records)
Y1_train = Y10_train[:, :1]
Y1_val = Y10_val[:, :1]
Y1_test = Y10_test[:, :1]

print("Train:", X_train.shape, Y1_train.shape, Y10_train.shape, Yseq_train.shape)
print("Valid:", X_val.shape, Y10_val.shape)
print("Test :", X_test.shape, Y10_test.shape)

# Sanity check: targets at the final input position must match exactly.
assert np.allclose(Yseq_train[:, -1, :], Y10_train)
assert np.allclose(Yseq_test[:, -1, :], Y10_test)
assert X_train.shape[1:] == (L, 1)
assert Yseq_train.shape[1:] == (L, H)

sample = 0
x_history = X_test[sample, :, 0]
y_future = Y10_test[sample]
plt.figure(figsize=(10, 3))
plt.plot(np.arange(-L + 1, 1), x_history, marker=".", label="observed history")
plt.plot(np.arange(1, H + 1), y_future, marker="o", label="next 10 actual values")
plt.axvline(0, linestyle="--", color="gray", label="prediction time")
plt.xlabel("Time relative to prediction time (samples)")
plt.ylabel("Sensor value")
plt.legend()
plt.tight_layout()
plt.show()

> **Q1.** At what time is the prediction made? Why must the 10 values in the future target not appear in the input?

**Your answer:** _Write your interpretation here._

> **Q2.** Explain the three dimensions of `X_train`. Why does the final dimension equal 1?

**Your answer:** _Write your interpretation here._

## 2. One-step forecasting: baselines before RNNs (15–20 min)

We predict **one future sensor value** from the previous 50. Use MSE consistently:

$$\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat y_i)^2.$$

First compare: (a) **persistence**: predict the last observed value; (b) **Ridge regression**: learn a linear combination of the 50 lagged samples. Ridge uses regularization to reduce unstable coefficients.

In [ ]:
def mse(y_true, y_pred):
    """Return a single mean squared error over all examples and horizons."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    assert y_true.shape == y_pred.shape, f"Shape mismatch: {y_true.shape} vs {y_pred.shape}"
    return float(np.mean((y_true - y_pred) ** 2))

# Persistence: X is [N, 50, 1], so [:, -1, 0] selects the last scalar.
pred_naive_1 = X_test[:, -1, 0:1]  # shape [N, 1]

# Ridge is a tabular model: flatten each 50-step window into 50 columns.
X_flat_train = X_train.reshape(len(X_train), L)
X_flat_val = X_val.reshape(len(X_val), L)
X_flat_test = X_test.reshape(len(X_test), L)

ridge_1 = Ridge(alpha=1.0)
ridge_1.fit(X_flat_train, Y1_train)
pred_ridge_1 = ridge_1.predict(X_flat_test).reshape(-1, 1)

one_step_results = {
    "Persistence": mse(Y1_test, pred_naive_1),
    "Ridge over 50 lags": mse(Y1_test, pred_ridge_1),
}
for name, score in one_step_results.items():
    print(f"{name:24s} test MSE = {score:.5f}")

> **Q3.** Are the values in the Ridge input different from those in the RNN input? What is different?

**Your answer:** _Write your interpretation here._

## 3. Train a SimpleRNN (20–25 min)

At each time step, a SimpleRNN updates its state:

$$h_t=\tanh(W_x x_t+W_h h_{t-1}+b).$$

The same weights are reused at **every** time step. The final hidden state is then mapped to a one-step forecast by `Dense(1)`.

For the lab, `units=16` means the hidden state has **16 components**. A single-neuron RNN (`units=1`) is also included as a deliberately small model.

For a SimpleRNN with input dimension d and hidden dimension U, the recurrent-layer parameter count is:

$$dU+U^2+U.$$

The Dense output layer adds `U + 1` parameters. This total does **not** depend on the sequence length L.

In [ ]:
def build_one_step_rnn(units=16):
    """A recurrent sequence-to-vector predictor: [N,L,1] -> [N,1]."""
    model = keras.Sequential([
        keras.Input(shape=(None, 1)),          # None allows variable input length
        keras.layers.SimpleRNN(units),         # returns final state [N, units]
        keras.layers.Dense(1),                # maps state -> one numeric forecast
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005, global_clipnorm=1.0),
        loss="mse",
    )
    return model

rnn_one = build_one_step_rnn(units=1)
rnn_sixteen = build_one_step_rnn(units=16)
print("One-unit RNN parameters:", rnn_one.count_params())
print("16-unit RNN parameters:", rnn_sixteen.count_params())
rnn_sixteen.summary()

# We train only on training windows and use the validation set to monitor fitting.
hist_one = rnn_one.fit(X_train, Y1_train, validation_data=(X_val, Y1_val),
                       epochs=10, batch_size=64, verbose=0)
hist_sixteen = rnn_sixteen.fit(X_train, Y1_train,
                               validation_data=(X_val, Y1_val),
                               epochs=12, batch_size=64, verbose=0)

one_step_results["SimpleRNN (1 unit)"] = mse(Y1_test, rnn_one.predict(X_test, verbose=0))
one_step_results["SimpleRNN (16 units)"] = mse(Y1_test, rnn_sixteen.predict(X_test, verbose=0))
print("ONE-STEP TEST MSE (lower is better)")
for name, score in one_step_results.items():
    print(f"{name:24s} {score:.5f}")

plt.figure(figsize=(8, 3))
plt.plot(hist_one.history["val_loss"], label="1 unit — validation")
plt.plot(hist_sixteen.history["val_loss"], label="16 units — validation")
plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.title("Training a small vs a wider SimpleRNN")
plt.legend()
plt.tight_layout()
plt.show()

> **Q4.** Calculate manually the parameter count for the 16-unit model (including Dense(1)). Does doubling L double the number of parameters?

**Your answer:** _Write your interpretation here._

> **Q5.** Which model has the lowest test MSE? Is a SimpleRNN guaranteed to beat persistence or Ridge?

**Your answer:** _Write your interpretation here._

**Optional architecture check (no additional training needed):** Stack two recurrent layers. The *first* must return a state at every time step, so that the second layer has a temporal sequence to process.

In [ ]:
stacked_rnn = keras.Sequential([
    keras.Input(shape=(None, 1)),
    keras.layers.SimpleRNN(12, return_sequences=True),  # output [N, L, 12]
    keras.layers.SimpleRNN(12),                         # output [N, 12]
    keras.layers.Dense(1),                              # output [N, 1]
])
print("Stacked model output for 3 windows:", stacked_rnn(X_test[:3]).shape)
print("Stacked model parameter count:", stacked_rnn.count_params())

## 4. Why Backpropagation Through Time can struggle (10–15 min)

The loss at the end of a sequence depends on earlier hidden states through a chain of updates. **Backpropagation Through Time (BPTT)** computes gradients by applying the chain rule backwards through that chain.

For an *illustrative simplified linear recurrence*, let

$$h_t=a h_{t-1}+x_t.$$

Then the sensitivity of a later state to an earlier state is

$$\left|\frac{\partial h_L}{\partial h_{L-k}}\right|=|a|^k.$$

This is **not** the complete SimpleRNN gradient: with `tanh`, each step also contributes a derivative factor. The simplified expression helps us see how repeated multiplication can suppress or amplify gradients.

In [ ]:
steps_back = np.arange(0, 51)  # number of recurrent updates backward
plt.figure(figsize=(9, 4))
for a in [0.80, 1.00, 1.08]:
    # Repeated chain-rule factor for a simplified linear recurrence.
    sensitivity = np.abs(a) ** steps_back
    plt.semilogy(steps_back, sensitivity, label=f"|a| = {a:.2f}")
plt.xlabel("How many time steps backward?")
plt.ylabel("Gradient magnitude (log scale)")
plt.title("Illustration: repeated multiplication in BPTT")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Exploding gradients can cause large parameter changes. Gradient clipping
# limits their magnitude; it does NOT fix a gradient that is almost zero.
gradient = np.array([24.0, -18.0])
max_norm = 1.0
norm_before = np.linalg.norm(gradient)
clipped = gradient * min(1.0, max_norm / (norm_before + 1e-12))
print("Gradient before clipping:", gradient, "norm:", norm_before)
print("Gradient after clipping: ", clipped, "norm:", np.linalg.norm(clipped))

> **Q6.** For which value of |a| does the gradient vanish, and for which value does it explode? Does gradient clipping solve vanishing gradients?

**Your answer:** _Write your interpretation here._

## 5. Direct forecasting of the next 10 values (15–20 min)

A **direct multi-horizon** sequence-to-vector model reads the entire 50-step history and produces a 10-dimensional output:

$$[N,50,1]\rightarrow\text{SimpleRNN(16)}\rightarrow[N,16]\rightarrow\text{Dense(10)}\rightarrow[N,10].$$

Unlike recursive one-step prediction, the model does **not** feed its forecast at horizon 1 back as the input for horizon 2. We compare it with a **multi-horizon persistence** baseline (repeat the last observed value 10 times) and a direct Ridge regression baseline.

In [ ]:
# The persistence forecast repeats the latest observation across ALL 10 horizons.
pred_naive_10 = np.repeat(X_test[:, -1, 0:1], H, axis=1)

ridge_10 = Ridge(alpha=1.0)
ridge_10.fit(X_flat_train, Y10_train)
pred_ridge_10 = ridge_10.predict(X_flat_test)

# Direct RNN: the final hidden state produces a single forecast vector.
direct_rnn = keras.Sequential([
    keras.Input(shape=(None, 1)),
    keras.layers.SimpleRNN(16),              # [N,16], final state only
    keras.layers.Dense(H),                  # [N,10], all horizons at once
])
direct_rnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005, global_clipnorm=1.0),
    loss="mse",
)
hist_direct = direct_rnn.fit(X_train, Y10_train,
                             validation_data=(X_val, Y10_val),
                             epochs=16, batch_size=64, verbose=0)
pred_direct_10 = direct_rnn.predict(X_test, verbose=0)
assert pred_direct_10.shape == Y10_test.shape

results_10 = {
    "Persistence (10 horizons)": mse(Y10_test, pred_naive_10),
    "Ridge direct (10 horizons)": mse(Y10_test, pred_ridge_10),
    "SimpleRNN direct (10 horizons)": mse(Y10_test, pred_direct_10),
}
for name, score in results_10.items():
    print(f"{name:34s} test MSE = {score:.5f}")

# Display the entire 10-horizon forecast for one previously unseen window.
example = 5
plt.figure(figsize=(10, 3))
plt.plot(np.arange(-L+1, 1), X_test[example, :, 0], label="observed history")
plt.plot(np.arange(1, H+1), Y10_test[example], "ko-", label="actual future")
plt.plot(np.arange(1, H+1), pred_naive_10[example], "--", label="persistence")
plt.plot(np.arange(1, H+1), pred_direct_10[example], "o-", label="direct SimpleRNN")
plt.axvline(0, color="gray", linestyle=":")
plt.xlabel("Relative time (samples)")
plt.ylabel("Sensor value")
plt.title("A direct 10-step forecast")
plt.legend(ncol=4, fontsize=9)
plt.tight_layout()
plt.show()

> **Q7.** How does the direct RNN produce 10 forecasts without a recurrent decoder? Why is recursive error feedback absent?

**Your answer:** _Write your interpretation here._

## 6. A forecast at every input time step (20–25 min)

Slides 237–242 propose a different **training objective**: at each input position j, predict its next 10 samples.

For one input window:

- Input: `[L, 1]`.
- Targets: `[L, 10]`: at j=0 predict steps 1…10; at j=1 predict steps 2…11; ...; at j=L−1 predict the final 10-step horizon.
- Full batch: **[N, L, 10]**.

This is **not** an encoder–decoder that generates future values recursively. We still process only observed input values. We use `return_sequences=True` so that a representation is available at every observed time step. `Dense(10)` automatically applies to the final dimension at every step; a separate `TimeDistributed(Dense(10))` wrapper is not required in modern Keras.

Training minimizes errors at all time steps. **For comparison with the direct model, evaluation must use only predictions at the last input time step**.

In [ ]:
# A custom Keras metric used DURING training to report the error for the
# only output we will actually use at deployment: the FINAL input time step.
def last_step_mse(y_true, y_pred):
    """MSE over 10 horizons, evaluated only at the last input time step.

    Inputs [batch, L, H]; output [batch] so Keras can average it.
    """
    last_true = y_true[:, -1, :]         # [batch, 10]
    last_pred = y_pred[:, -1, :]         # [batch, 10]
    squared_error = keras.ops.square(last_true - last_pred)
    return keras.ops.mean(squared_error, axis=-1)  # one number per sample

seq_rnn = keras.Sequential([
    keras.Input(shape=(None, 1)),
    keras.layers.SimpleRNN(16, return_sequences=True),  # [N, L, 16]
    keras.layers.Dense(H),                            # [N, L, 10]
])
seq_rnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005, global_clipnorm=1.0),
    loss="mse",                    # MSE over EVERY observed time step and horizon
    metrics=[last_step_mse],        # diagnostic metric for the final forecast only
)
print("One forward pass gives shape:", seq_rnn(X_test[:2]).shape)
hist_seq = seq_rnn.fit(X_train, Yseq_train,
                       validation_data=(X_val, Yseq_val),
                       epochs=16, batch_size=64, verbose=0)

pred_at_every_step = seq_rnn.predict(X_test, verbose=0)
assert pred_at_every_step.shape == (len(X_test), L, H)

# At prediction time we keep only the output produced after reading all 50 inputs.
pred_seq_last = pred_at_every_step[:, -1, :]
assert pred_seq_last.shape == Y10_test.shape
results_10["SimpleRNN seq-to-seq (final)"] = mse(Y10_test, pred_seq_last)

print("TEN-HORIZON TEST MSE — SAME TEST SET, FINAL PREDICTIONS")
for name, score in results_10.items():
    print(f"{name:34s} {score:.5f}")

# Verify that the custom metric matches a direct NumPy calculation.
reported_test = seq_rnn.evaluate(X_test, Yseq_test, verbose=0, return_dict=True)
print("Keras test loss over ALL time steps:", round(reported_test["loss"], 6))
print("Keras last-step test metric:", round(reported_test["last_step_mse"], 6))
print("NumPy last-step test MSE:  ", round(mse(Y10_test, pred_seq_last), 6))
assert np.isclose(reported_test["last_step_mse"], mse(Y10_test, pred_seq_last), rtol=5e-4)

> **Q8.** Why is `seq_rnn.evaluate(...)["loss"]` not directly comparable to the direct model’s 10-horizon MSE?

**Your answer:** _Write your interpretation here._

> **Q9.** What is the difference between `return_sequences=False` and `True` in terms of the RNN output shapes? Does Dense(10) require `TimeDistributed` here?

**Your answer:** _Write your interpretation here._

## 7. Fair evaluation and engineering interpretation (10 min)

Do not report only one average score. Inspect *where* a model succeeds or fails: one-step versus longer horizons, isolated peaks, and changes between recording conditions. Remember that a more complex model must **earn** its added complexity.

Below, each horizon is scored separately, with all models evaluated against the **same test targets**.

In [ ]:
models_for_horizon = {
    "Persistence": pred_naive_10,
    "Ridge": pred_ridge_10,
    "Direct SimpleRNN": pred_direct_10,
    "Seq-to-seq SimpleRNN (last)": pred_seq_last,
}
plt.figure(figsize=(9, 4))
for model_name, predictions in models_for_horizon.items():
    horizon_mse = np.mean((predictions - Y10_test) ** 2, axis=0)
    plt.plot(np.arange(1, H+1), horizon_mse, marker="o", label=model_name)
plt.xlabel("Forecast horizon (samples ahead)")
plt.ylabel("Test MSE")
plt.title("Fair comparison: identical horizons and held-out recordings")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 3))
plt.plot(hist_direct.history["loss"], label="Direct: training MSE (final only)")
plt.plot(hist_direct.history["val_loss"], label="Direct: validation MSE")
plt.plot(hist_seq.history["last_step_mse"], label="Seq-to-seq: train last-step MSE")
plt.plot(hist_seq.history["val_last_step_mse"], label="Seq-to-seq: validation last-step MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE on comparable final forecasts")
plt.title("Do not compare unlike training objectives")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

> **Q10.** Which approach would you recommend for this dataset, considering both MSE and model complexity? Is the most complex always the best?

**Your answer:** _Write your interpretation here._

## Optional student extensions (only if time remains)

1. Change **L from 50 to 25 or 80**; restart the notebook and repeat. The number of RNN parameters stays the same, but runtime and available context change. (The Ridge number of input features *does* change.)
2. Change the number of SimpleRNN units from **16 to 4**. Look for underfitting; do not assume the result before measuring it.
3. Train a two-layer recurrent model by adding `SimpleRNN(16, return_sequences=True)` before the last recurrent layer. Explain why the first layer must return a sequence.
4. Change the forecast horizon **H from 10 to 5** and explain how the direct and sequence-to-sequence target/output shapes change.

### Final checklist

- [ ] I can explain `[N, L, d]`, `[N, H]`, and `[N, L, H]`.
- [ ] I know the difference between a recurrent hidden state and a manually designed rolling statistic.
- [ ] I can explain why RNN parameters are reused across time and how BPTT follows dependencies backward.
- [ ] I can distinguish a direct multi-horizon prediction from recursive multi-step prediction.
- [ ] I know why the sequence-to-sequence **training loss** and **final-step evaluation metric** differ.
- [ ] I have compared every model on the same held-out recordings.

*Lecture mapping: slides 1–61 (representation and output structures), 62–95 (BPTT and gradient stability), and 230–242 (Keras forecasting architectures, sequence-to-sequence targets, final-step metric).*